In [73]:
import random

suits = ('Hearts','Diamonds','Spades','Clubs')
ranks = ('Two', 'Three', 'Four', 'Five', 'Six', 'Seven', 'Eight', 'Nine', 'Ten', 'Jack', 'Queen', 'King', 'Ace')
values = {'Two':2, 'Three':3, 'Four':4, 'Five':5, 'Six':6, 'Seven':7, 'Eight':8, 'Nine':9, 'Ten':10, 'Jack':10,'Queen':10, 'King':10, 'Ace':11}

playing = True

In [74]:
class Card:
    
    def __init__(self, suit, rank):
        self.suit = suit
        self.rank = rank
        
    def __str__(self):
        return f'{self.rank} of {self.suit}'

In [75]:
class Deck:
    
    def __init__(self):
        self.deck = []  # start with an empty list
        for suit in suits:
            for rank in ranks:
                my_card = Card(suit,rank)
                self.deck.append(my_card)
    
    def __str__(self):
        return '\n'.join(f'{card.rank} of {card.suit}' for card in self.deck)

    def shuffle(self):
        random.shuffle(self.deck)
        
    def deal(self):
        return self.deck.pop()

In [76]:
class Hand:
    def __init__(self):
        self.cards = []  # start with an empty list as we did in the Deck class
        self.value = 0   # start with zero value
        self.aces = 0    # add an attribute to keep track of aces
        
        self.bet = 0
        self.insurance = 0
        self.is_active = True
        self.is_split_hand = False 
        self.doubled = False 
    
    def add_card(self,card):
        #card passed in 
        #from Deck.deal() --> single Card(suit,rank)
        self.cards.append(card)
        self.value += values[card.rank]
        
        #track aces
        if card.rank == 'Ace':
            self.aces += 1
        
        self.adjust_for_ace() #for automatic ace conversions 
    
    def adjust_for_ace(self):
        
        #IF MY TOTAL VALUE IS > 21 AND I STILL HAVE AN ACE 
        #THEN CHANGE MY ACE TO BE A 1 INSTEAD OF 11
        while self.value > 21 and self.aces:
            self.value -= 10
            self.aces -= 1
            
    def is_blackjack(self):
        return len(self.cards) == 2 and self.value == 21

In [77]:
class Chips:
    
    def __init__(self,total=100):
        self.total = total  # This can be set to a default value or supplied by a user input
        self.bet = 0
        
    def win_bet(self):
        self.total += self.bet
    
    def lose_bet(self):
        #self.total -= self.bet
        pass

In [78]:
def take_bet(chips):
    print(f'\n💰 Current Balance: {chips.total} chips')
    while True:
        try:
            bet = int(input(f'You have {chips.total} chips. How many would you like to bet? '))
        except:
            print('Sorry please provide an integer!')
            continue
        if bet <= 0:
            print('Bet must be positive.')
        elif bet > chips.total:
                print(f'Sorry, you only have {chips.total} chips!')
        else:
            chips.bet = bet
            chips.total -= bet #the chips you bet are "on the table" and no longer yours until the round ends.
            return bet 

In [79]:
def offer_insurance(hand, chips, dealer_upcard):
    # offer insurance only when dealer upcard is ace
    if dealer_upcard.rank != 'Ace':
        return 
    #insurance is allowed only up to half of your original bet 
    max_ins = hand.bet // 2
    if max_ins == 0:
        return 
    print(f'\n🛡️  Dealer shows an Ace! Insurance available (pays 2:1)')
    print(f' You can insure up to {max_ins} chips')
    
    while True:
        
        try:
            ins = int(input('Enter insurance amount (0 to skip): '))
        except:
            print('Enter a valid number.')
            continue
            
        if 0 <= ins <= max_ins and ins <= chips.total:
            hand.insurance = ins # split hands can also have insurance
            chips.total -= ins # insurance is like a second bet placed on the table
            if ins > 0:
                print(f'✓ Insurance placed: {ins} chips')
            break
        else:
            print(f'Invalid amount. Must be 0..{min(max_ins, chips.total)}.')
        

In [80]:
def can_split(hand,chips):
    return len(hand.cards) == 2 and hand.cards[0].rank == hand.cards[1].rank and chips.total >= hand.bet
    # chips.total >= hand.bet: checks if the player has enough chips for a second bet 

In [81]:
def do_split(hand,deck,chips):
    print('\n✂️  Splitting hand...')
    chips.total -= hand.bet #deduct second bet
    
    h1 = Hand()
    h2 = Hand()
    
    h1.add_card(hand.cards[0])
    h2.add_card(hand.cards[1])
    
    h1.bet = hand.bet
    h2.bet = hand.bet
    
    h1.is_split_hand = True
    h2.is_split_hand = True
    
    h1.add_card(deck.deal())
    h2.add_card(deck.deal())
    
    return [h1, h2]

In [82]:
def do_double(hand,deck,chips):
    
    if chips.total < hand.bet:
        print('Not enough chips to double down.')
        return False 
    
    print('\n⬆️  Doubling down...')
    chips.total -= hand.bet
    hand.bet *= 2
    hand.doubled = True
    hand.add_card(deck.deal()) # deals ONE card
    hand.is_active = False # auto-stand after double
    return True

In [83]:
def hit(deck,hand):
    
    hand.add_card(deck.deal())
    if hand.value > 21:
        hand.is_active = False

In [84]:
# def hit_or_stand(deck,hand):
#     global playing  # to control an upcoming while loop
    
#     while True:
#         x = input('\nHit or Stand? Enter h or s!')
        
#         if x[0].lower() == 'h':
#             hit(deck,hand)
            
#         elif x[0].lower() == 's':
#             print("\nPlayer Stands, Dealer's turn")
#             playing = False 
    
#         else:
#             print('Sorry, I did not understand that, Please enter h or s only!')
#             continue
            
#         break

In [85]:
def show_some(player_hands,dealer_hand):
    
    #Show only one of the dealer's cards
    print("\n" + "-"*50)
    print("\nDealer's hand: ")
    print(' <Hidden card>')
    # Show only dealer's upcard (dealer.cards[0] is hidden, dealer.cards[1] shown)
    if len(dealer_hand.cards) >= 2:
        print(f' {dealer_hand.cards[1]}')
    
    #Show all of the player's cards
    print("\nPlayer's hands: ")
    for idx, h in enumerate(player_hands, 1):
        status = '(active)' if h.is_active else ''
        cards = ', '.join(str(c) for c in h.cards)
        print(f'Hand {idx}: {cards} | Value: {h.value} | Bet: {h.bet} chips {status}')
        print('-'*50)
        
def show_all(player_hands,dealer_hand):
    
    #Show all of the dealer's cards
    print("\n" + "-"*50)
    print("\nDealer's full hand: ")
    print(f' {", ".join(str(c) for c in dealer_hand.cards)}')
    #print("Dealer's hand: ",*dealer.cards,sep='')
    
    #calculate and display value (J+K==20)
    print(f' Value: {dealer_hand.value}')
    
    #Show all of the player's cards
    print("\nPlayer's hand: ")
    for idx, h in enumerate(player_hands, 1):
        cards = ', '.join(str(c) for c in h.cards)
        print(f'Hand {idx}: {cards} | Value: {h.value} | Bet: {h.bet}')
    print("-"*50 + "\n")

In [86]:
# def player_busts(player,dealer,chips):
#     print('\nPLAYER BUSTS')
#     chips.lose_bet()

# def player_wins(player,dealer,chips):
#     print('\nPLAYER WINS')
#     chips.win_bet()

# def dealer_busts(player,dealer,chips):
#     print('\nPLAYER WINS! DEALER BUSTED')
#     chips.win_bet()
    
# def dealer_wins(player,dealer,chips):
#     print('\nDEALER WINS!')
#     chips.lose_bet()
    
# def push():
#     print('\nPlayer and Dealer tie! PUSH')

In [87]:
def is_blackjack(hand):
    return hand.is_blackjack()

In [88]:
# round_no = 0
# while True:
    
#     # Print an opening statement
#     round_no += 1
#     print('WELCOME TO BLACKJACK!')
#     print(f'\nROUND {round_no} BEGINS!')
    
#     # Create & shuffle the deck, deal two cards to each player
#     my_deck = Deck()
#     my_deck.shuffle()
    
#     player_hands = [Hand()]
#     dealer_hand = Hand()
    
#     for card in range(2):
#         player_hand.add_card(my_deck.deal())
#         dealer_hand.add_card(my_deck.deal())
    
#     # Set up the Player's chips
#     player_chips = Chips()
    
#     # Prompt the Player for their bet
#     take_bet(player_chips)
    
#     # Show cards (but keep one dealer card hidden)
#     show_some(player_hand,dealer_hand)
    
#     while playing:  # recall this variable from our hit_or_stand function
        
#         # Prompt for Player to Hit or Stand
#         hit_or_stand(my_deck,player_hand)
        
#         # Show cards (but keep one dealer card hidden)
#         show_some(player_hand,dealer_hand)
        
#         # If player's hand exceeds 21, run player_busts() and break out of loop
#         if player_hand.value > 21:
#             player_busts(player_hand,dealer_hand,player_chips)
#             break

#     # If Player hasn't busted, play Dealer's hand until Dealer reaches 17
#     if player_hand.value <= 21:
        
#         while dealer_hand.value < 17:
#             hit(my_deck,dealer_hand)
    
#         # Show all cards
#         show_all(player_hand,dealer_hand)
        
#         # Run different winning scenarios
#         if dealer_hand.value > 21:
#             dealer_busts(player_hand,dealer_hand,player_chips)
#         elif player_hand.value > dealer_hand.value:
#             player_wins(player_hand,dealer_hand,player_chips)
#         elif player_hand.value < dealer_hand.value:
#             dealer_wins(player_hand,dealer_hand,player_chips)
#         else:
#             push()
    
#     # Inform Player of their chips total 
#     print(f'\nTotal number of Chips: {player_chips.total}')
    
#     # Ask to play again
#     play = input('\nDo you want to play again? y(yes) or n(no)?')
#     if play[0].lower() == 'y':
#         playing = True
#         continue
#     else:
#         print('\nThank you for playing!')
#         break

In [91]:
# --- Main game logic for a single round (console) ---
def play_round(chips):
    deck = Deck()
    deck.shuffle()
    
    dealer_hand = Hand()
    player_hands = [Hand()]
    
    bet = take_bet(chips)
    player_hands[0].bet = bet
    
    for _ in range(2):
        player_hands[0].add_card(deck.deal())
        dealer_hand.add_card(deck.deal())
    
    # Offer insurance if dealer upcard is Ace (we show dealer.cards[1] to player in console)
    offer_insurance(player_hands[0], chips, dealer_hand.cards[1])
    
    # check for naturals
    player_natural = is_blackjack(player_hands[0])
    dealer_natural = is_blackjack(dealer_hand)
    
    if dealer_natural:
        print('\nDealer has natural Blackjack!')
        
        # Resolve insurance first
        if player_hands[0].insurance > 0: 
            # Insurance is offered immediately after the initial deal i.e before splitting happens 
            # insurance pays 2:1; since we deducted insurance earlier, add 3x to net (return + profit)
            chips.total += player_hands[0].insurance * 3
            print(f'Insurance wins. Paid {player_hands[0].insurance * 2} chips profit.')
            
        if player_natural:
            print("Push — both have Blackjack. Bet returned.")
            chips.total += player_hands[0].bet
        else:
            print('Dealer wins Blackjack')
            # player's bet is already deducted
        return 

    if player_natural:
        print('\nPlayer has a natural Blackjack! Pays 3:2')
        # Blackjack pays 3:2 : since bet was deducted at start, we add back floor(2.5 * bet)
        payout = (player_hands[0].bet * 5)//2
        chips.total += payout 
        return 
    
    # Show insurance loss if dealer doesn't have blackjack
    if player_hands[0].insurance > 0:
        print(f'\n❌ Insurance loses! Dealer does not have Blackjack. (-{player_hands[0].insurance} chips)')
    
    i = 0
    while i < len(player_hands):
        
        hand = player_hands[i]
        
        if hand.value > 21:
            hand.is_active = False
            
        while hand.is_active:
            show_some(player_hands, dealer_hand)
            
            options = ['[H]it', '[S]tand']
            allow_double = len(hand.cards) == 2 and chips.total >= hand.bet
            allow_split = can_split(hand, chips)
            
            if allow_double:
                options.append('[D]ouble')
                
            if allow_split:
                options.append('S[P]lit')
            
            print(f"\nOptions: {' | '.join(options)}")
            choice = input('Enter your choice (H/S/D/P): ').strip().lower()
            if not choice:
                print('Please enter a choice.')
                continue
            
            c = choice[0]
            
            if c == 'h':
                hit(deck, hand)
                print(f'Hit! New card: {hand.cards[-1]}')
                if hand.value > 21:
                    print(f'Hand busted! (Value: {hand.value})')
                    hand.is_active = False
                    
            elif c == 's':
                print(f'Standing on {hand.value}')
                hand.is_active = False
            
            elif c == 'd' and allow_double:
                do_double(hand, deck, chips)
                print(f'New card: {hand.cards[-1]} | Hand value: {hand.value}')
                if hand.value > 21:
                    print(f'Busted after doubling!')
                
            elif c == 'p' and allow_split:
                new_hands = do_split(hand, deck, chips)
                player_hands[i:i+1] = new_hands
                hand = player_hands[i] # we are now working with the first new split hand, which replaced the old one.
                # hand = [Hand_A] --> hand = [Hand_1, Hand_2]
                #                               i 
                
            else:
                print('Invalid option or not allowed this time!')
                continue
                
        i += 1
        
        # All player hands completed, Dealer plays if at least one hand did not bust
        if any(h.value <= 21 for h in player_hands):
            print('\n' + '-'*50)
            print('\nDealer reveals hole card...')
            show_all(player_hands, dealer_hand)
            # Dealer hits until value >= 17; stands on soft 17
            while dealer_hand.value < 17:
                print('Dealer hits...')
                dealer_hand.add_card(deck.deal())
                print(f"Dealer now: {', '.join(str(c) for c in dealer_hand.cards)} | Value: {dealer_hand.value}")
                if dealer_hand.value > 21:
                    print('Dealer busts!')
                    break
            if dealer_hand.value <= 21:
                print(f'Dealer stands on {dealer_hand.value}')
        # All player hands busted; still reveal dealer
        else:
            print('\nAll player hands busted!')
            show_all(player_hands, dealer_hand)
            
        # Resolution: for each player hand separately
        print('\n' + '-'*50)
        print('RESULTS:')
        print('-'*50)
        for idx, hand in enumerate(player_hands, 1):
            print(f'\nResolving Hand {idx}: {", ".join(str(c) for c in hand.cards)} | Value: {hand.value} | Bet: {hand.bet}')
            # Resolve insurance (if dealer had natural earlier would have been handled)
            # Note: insurance already paid/collected when dealer natural was checked above
            
            if hand.value > 21:
                print('Player busted. Dealer wins this hand!')
                # bet already deducted at the start
            elif dealer_hand.value > 21:
                # Dealer busted => player wins amount equal to bet
                print('Dealer busted. Player wins.')
                chips.total += hand.bet * 2 # return bet+winnings
            else:
                if hand.value > dealer_hand.value:
                    print('Player hand beats dealer. Player wins.')
                    chips.total += hand.bet * 2
                elif hand.value < dealer_hand.value:
                    print('Dealer wins this hand.')
                    # player's bet is already deducted
                else:
                    print(f'Push (tie at {hand.value}). Bet returned.')
                    chips.total += hand.bet

In [90]:
# --- Game Loop ---
def main():
    
    print('♠️ ♥️ ♦️ ♣️  WELCOME TO BLACKJACK!  ♣️ ♦️ ♥️ ♠️')
    player_chips = Chips(total=500)
    round_no = 0
    
    while True:
        round_no += 1
        print('\n' + '-'*50)
        print(f'ROUND {round_no}')
        print('-'*50)
        
        play_round(player_chips)
        
        print(f'\n💰 Balance: {player_chips.total} chips')
        
        if player_chips.total <= 0:
            print("\nYou're out of chips! Game Over.")
            break
            
        again = input('\nPlay another round? (y/n): ').strip().lower()
        if again and again[0] == 'y':
            continue
        else:
            print('\n' + '-'*50)
            print('Thanks for playing!')
            print('-'*50)
            break
if __name__ == '__main__':
    main()

♠️ ♥️ ♦️ ♣️  WELCOME TO BLACKJACK!  ♣️ ♦️ ♥️ ♠️

--------------------------------------------------
ROUND 1
--------------------------------------------------

💰 Current Balance: 500 chips
You have 500 chips. How many would you like to bet? 
Sorry please provide an integer!
You have 500 chips. How many would you like to bet? 200

🛡️  Dealer shows an Ace! Insurance available (pays 2:1)
 You can insure up to 100 chips
Enter insurance amount (0 to skip): 100
✓ Insurance placed: 100 chips

--------------------------------------------------

Dealer's hand: 
 <Hidden card>
 Ace of Clubs

Player's hands: 
Hand 1: Jack of Hearts, Five of Clubs | Value: 15 | Bet: 200 chips (active)
--------------------------------------------------

Options: [H]it | [S]tand | [D]ouble
Enter your choice (H/S/D/P): h
Hit! New card: Three of Diamonds

--------------------------------------------------

Dealer's hand: 
 <Hidden card>
 Ace of Clubs

Player's hands: 
Hand 1: Jack of Hearts, Five of Clubs, Three of Di